# 🏗️ CL-SDRG Phase 2: FGA Architecture & VRAM Validation

**Target:** Google Colab T4 GPU (< 2.5 GB peak VRAM)

This notebook:
1. Loads and **freezes** the `multilingual-E5-base` encoder backbone
2. Constructs the **Feature Gating Agent (FGA)** MLP module
3. Builds the **gated fusion** mechanism and **veracity classifier** head
4. Implements the **counterfactual perturbation** generator
5. Runs a forward-pass **dry-run** to validate VRAM < 2.5 GB

**⏱ Estimated Runtime:** ~2 minutes

## 1. Environment Setup

In [ ]:
!pip install -q torch transformers

In [ ]:
import sys, logging, random
import torch
import torch.nn as nn
import torch.nn.functional as F

# Logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter('[%(asctime)s] %(levelname)-8s %(message)s', datefmt='%H:%M:%S'))
logger.addHandler(handler)

# Seed
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    logging.info(f'Using GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    logging.info('No GPU — using CPU')

def fmt(n): return f'{n:,}'

# Config
BASE_ENCODER_NAME = 'intfloat/multilingual-e5-base'
EMBEDDING_DIM = 768
MAX_SEQ_LENGTH = 128
FGA_HIDDEN_DIM = 256
FGA_INPUT_DIM = EMBEDDING_DIM * 3
FGA_NUM_GATES = 3
CLASSIFIER_HIDDEN_DIM = 256
CLASSIFIER_DROPOUT = 0.1
NUM_CLASSES = 3
PHYSICAL_BATCH_SIZE = 16

print(f'✅ Config loaded | Device: {device}')

## 2. Frozen Base Encoder

In [ ]:
from transformers import AutoModel, AutoTokenizer

class FrozenEncoder(nn.Module):
    def __init__(self, model_name=BASE_ENCODER_NAME):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)
        for param in self.encoder.parameters():
            param.requires_grad = False
        self.encoder.eval()
        self.embedding_dim = self.encoder.config.hidden_size
        logging.info(f'Frozen encoder loaded: {model_name}')
        logging.info(f'  Embedding dim: {self.embedding_dim}')
        logging.info(f'  Frozen params: {fmt(sum(p.numel() for p in self.encoder.parameters()))}')

    @torch.no_grad()
    def encode(self, texts, device):
        prefixed = [f'query: {t}' for t in texts]
        tokens = self.tokenizer(prefixed, max_length=MAX_SEQ_LENGTH,
                                padding=True, truncation=True, return_tensors='pt').to(device)
        outputs = self.encoder(**tokens)
        mask = tokens['attention_mask'].unsqueeze(-1).float()
        embs = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        return embs

print('✅ FrozenEncoder defined')

## 3. Feature Gating Agent (FGA)

In [ ]:
class FeatureGatingAgent(nn.Module):
    """MLP: [E_q; E_s; E_t] → Linear → ReLU → Linear → Sigmoid → (α_q, α_s, α_t)"""
    def __init__(self, input_dim=FGA_INPUT_DIM, hidden_dim=FGA_HIDDEN_DIM,
                 embedding_dim=EMBEDDING_DIM, num_gates=FGA_NUM_GATES):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.gate_network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, embedding_dim * num_gates),
            nn.Sigmoid(),
        )
        logging.info(f'FGA: {fmt(sum(p.numel() for p in self.parameters() if p.requires_grad))} trainable params')

    def forward(self, e_q, e_s, e_t):
        concat = torch.cat([e_q, e_s, e_t], dim=-1)
        gates = self.gate_network(concat)
        return gates.split(self.embedding_dim, dim=-1)


class GatedFusion(nn.Module):
    """E_gated = α_q⊙E_q + α_s⊙E_s + α_t⊙E_t"""
    def forward(self, e_q, e_s, e_t, a_q, a_s, a_t):
        return a_q * e_q + a_s * e_s + a_t * e_t


class VeracityClassifier(nn.Module):
    def __init__(self, input_dim=EMBEDDING_DIM, hidden_dim=CLASSIFIER_HIDDEN_DIM,
                 num_classes=NUM_CLASSES, dropout=CLASSIFIER_DROPOUT):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )
        logging.info(f'Classifier: {fmt(sum(p.numel() for p in self.parameters() if p.requires_grad))} trainable params')

    def forward(self, e_gated):
        return self.classifier(e_gated)


class CounterfactualPerturbation:
    def __init__(self, speakers):
        self.all_speakers = list(set(speakers))
    def perturb(self, originals):
        return [random.choice([s for s in self.all_speakers if s != o] or [o]) for o in originals]

print('✅ FGA, GatedFusion, Classifier, Perturbation defined')

## 4. Assembled CL-SDRG Model

In [ ]:
class CLSDRG(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = FrozenEncoder()
        self.fga = FeatureGatingAgent()
        self.fusion = GatedFusion()
        self.classifier = VeracityClassifier()

    def forward(self, claims, speakers, dates, device):
        e_q = self.encoder.encode(claims, device)
        e_s = self.encoder.encode(speakers, device)
        e_t = self.encoder.encode(dates, device)
        a_q, a_s, a_t = self.fga(e_q, e_s, e_t)
        e_gated = self.fusion(e_q, e_s, e_t, a_q, a_s, a_t)
        logits = self.classifier(e_gated)
        return logits, (a_q, a_s, a_t)

print('✅ CLSDRG assembled')

## 5. VRAM Dry-Run Test

In [ ]:
import time

print('='*70)
print('  VRAM Dry-Run Test')
print('='*70)

if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

# Build model
t0 = time.time()
model = CLSDRG().to(device)
print(f'Model init: {time.time()-t0:.1f}s')

# Parameter summary
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - trainable
print(f'\n  Parameters:')
print(f'    Total:     {fmt(total)}')
print(f'    Frozen:    {fmt(frozen)}')
print(f'    Trainable: {fmt(trainable)} ({trainable/total*100:.2f}%)')

# Dummy forward pass
bs = PHYSICAL_BATCH_SIZE
dummy_claims = [f'This is a test claim number {i} about politics' for i in range(bs)]
dummy_speakers = [f'Speaker {i}' for i in range(bs)]
dummy_dates = [f'2023-01-{(i%28)+1:02d}' for i in range(bs)]

t0 = time.time()
with torch.no_grad():
    if device.type == 'cuda':
        with torch.cuda.amp.autocast():
            logits, gates = model(dummy_claims, dummy_speakers, dummy_dates, device)
    else:
        logits, gates = model(dummy_claims, dummy_speakers, dummy_dates, device)
print(f'\nForward pass: {time.time()-t0:.1f}s')
print(f'  Logits:  {logits.shape}')
print(f'  Gate α_q: {gates[0].shape}')

# Memory report
if device.type == 'cuda':
    alloc = torch.cuda.memory_allocated() / 1024**2
    peak = torch.cuda.max_memory_allocated() / 1024**2
    print(f'\n  GPU Memory:')
    print(f'    Allocated: {alloc:.1f} MB')
    print(f'    Peak:      {peak:.1f} MB ({peak/1024:.2f} GB)')
    if peak/1024 < 2.5:
        print(f'\n  ✅ VRAM TEST PASSED: {peak/1024:.2f} GB < 2.5 GB budget')
    else:
        print(f'\n  ⚠️  VRAM TEST FAILED: {peak/1024:.2f} GB ≥ 2.5 GB budget')
else:
    print('  (CPU-only run — VRAM test skipped)')

# Counterfactual test
perturb = CounterfactualPerturbation(dummy_speakers)
perturbed = perturb.perturb(dummy_speakers)
diff = sum(1 for o, p in zip(dummy_speakers, perturbed) if o != p)
print(f'\n  Counterfactual: {diff}/{bs} speakers changed')

print('\n' + '='*70)
print('  PHASE 2 COMPLETE — Architecture validated')
print('='*70)
print('\n📋 Copy VRAM results above and share back for Phase 3.')